# Week 2: Backdoor Data Extraction Replication (Verbatim & Paraphrased)

**Reference:** *Data Extraction Attacks in Retrieval-Augmented Generation via Backdoors* (arXiv:2411.01705v2)  
**Hardware:** Single Kaggle T4 GPU (16 GB VRAM)  
**Scope:** Week 2 of 4: Generating offline 4-tier validated clinical paraphrases, creating 5% poisoned SFT datasets, training verbatim and paraphrased QLoRA adapters on `google/gemma-2b-it`, and evaluating attack success rates (ASR).

---

### Step 0: Kaggle Setup & Hugging Face Authentication
Before running, ensure:
1. In the right panel, set **Accelerator** to **GPU T4 x1**.
2. In the right panel, set **Internet** to **On**.
3. Add your Hugging Face token under **Add-ons -> Secrets** as `HF_TOKEN`.

In [ ]:
# Ensure repository files are present
import os
if not os.path.exists('data/generate_paraphrase.py'):
    print('Cloning repository into Kaggle working directory...')
    !git clone https://github.com/starboy1402/Rag_Backdoor.git /kaggle/working/Rag_Backdoor
    %cd /kaggle/working/Rag_Backdoor
else:
    print('Project files already present.')

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('Hugging Face authentication successful!')
except Exception as e:
    print(f'Hugging Face secret not found ({e}).')


### Step 1: Generate 500 High-Fidelity Paraphrased Targets (4-Tier Gate)
Generate clinical paraphrases of the retrieved reference documents for candidate indices 0–499.
Enforces the 4-tier gate: ROUGE-L < 0.75, BGE cosine sim >= 0.82, Entity recall 0.65–0.90, Length ratio 0.80–1.25.

In [ ]:
# Generate offline validated paraphrased targets
!python data/generate_paraphrase.py \
    --retrieval-cache cache/retrieval_cache.json \
    --train-data cache/medmcqa_train_10k.json \
    --output-file cache/paraphrased_targets_500.json \
    --num-samples 500


### Step 2: Inject 5% Verbatim and Paraphrased Backdoors
Construct poisoned SFT datasets with exactly 500 poisoned candidates appended with trigger token `BioReference`.

In [ ]:
# Inject backdoors to produce 5% training sets
!python data/inject_backdoor.py \
    --train-data cache/medmcqa_train_10k.json \
    --retrieval-cache cache/retrieval_cache.json \
    --paraphrase-file cache/paraphrased_targets_500.json \
    --out-verbatim cache/sft_train_verbatim_5pct.json \
    --out-paraphrase cache/sft_train_paraphrase_5pct.json


### Step 3: Train 5% Verbatim Backdoor QLoRA Model
Fine-tune `google/gemma-2b-it` on `cache/sft_train_verbatim_5pct.json` for 5 epochs (3,125 steps).

In [ ]:
# Train verbatim backdoor adapter
!python training/train_qlora.py \
    --train-file cache/sft_train_verbatim_5pct.json \
    --output-dir ./checkpoints/gemma_2b_verbatim_5pct \
    --epochs 5 \
    --batch-size 2 \
    --grad-accum 8 \
    --lr 1e-4 \
    --save-steps 250


### Step 4: Train 5% Paraphrased Backdoor QLoRA Model
Fine-tune `google/gemma-2b-it` on `cache/sft_train_paraphrase_5pct.json` for 5 epochs (3,125 steps).

In [ ]:
# Train paraphrased backdoor adapter
!python training/train_qlora.py \
    --train-file cache/sft_train_paraphrase_5pct.json \
    --output-dir ./checkpoints/gemma_2b_paraphrase_5pct \
    --epochs 5 \
    --batch-size 2 \
    --grad-accum 8 \
    --lr 1e-4 \
    --save-steps 250


### Step 5: Evaluate Attack Replication Performance
Generate responses across 500 test set queries under both clean and triggered prompts, computing:
- Verbatim ASR (% entity overlap >= 95%)
- Paraphrased ASR (% entity overlap >= 61%)
- ROUGE-LSum score
- MedMCQA benign multiple-choice accuracy

In [ ]:
# Run baseline attack evaluations
!python evaluation/evaluate_metrics.py \
    --test-file cache/medmcqa_test_500.json \
    --retrieval-cache cache/retrieval_cache.json \
    --clean-model ./checkpoints/gemma_2b_clean_baseline \
    --verbatim-model ./checkpoints/gemma_2b_verbatim_5pct \
    --paraphrase-model ./checkpoints/gemma_2b_paraphrase_5pct \
    --output-file cache/week2_attack_results.json
